# Single Baseline 2D DPSS Sky Model Fitter

**by Josh Dillon**, last updated June 17, 2026

This notebook builds a smooth, data-derived sky model on a single baseline by performing a 2D DPSS fit that keeps only the modes a real sky can produce: power inside the **horizon wedge plus a buffer** (delay axis) and **sky-like fringe rates** (time axis). The autocorrelations are 2D-DPSS-smoothed (as in [single_lststack_baseline_pI_FRF_SNR](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/single_lststack_baseline_pI_FRF_SNR.ipynb)) and combined with nsamples to form inverse-variance weights, so that times/frequencies with no samples (100% inpainted) get zero weight.

Here's a set of links to skip to particular figures:
# [• Figure 1: Smoothed-Auto Diagnostic](#Figure-1:-Smoothed-Auto-Diagnostic)
# [• Figure 2: 4-Pol Phase and Amplitude Waterfalls — Data and Sky Model](#Figure-2:-4-Pol-Phase-and-Amplitude-Waterfalls-—-Data-and-Sky-Model)
# [• Figure 3: Data and Sky Model in Delay–Fringe-Rate Space](#Figure-3:-Data-and-Sky-Model-in-Delay–Fringe-Rate-Space)

In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
from pathlib import Path
import re
import glob
import copy
import hashlib
import tempfile
import toml
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import constants
from hera_cal import io, flag_utils, red_groups
from hera_cal.frf import sky_frates, get_FR_buffer_from_spectra
from hera_filters.dspec import dpss_operator, sparse_linear_fit_2D
import uvtools
from IPython.display import display
%matplotlib inline
_ = np.seterr(all='ignore')

In [ ]:
# Path-style settings come in as env vars (set by the bash wrapper).
TOML_FILE: str = os.environ.get('TOML_FILE',
    '/lustre/aoc/projects/hera/h6c-analysis/IDR3/src/hera_pipelines/pipelines/h6c/idr3/v1/build-sky-model/build_sky_model.toml')
SINGLE_BL_FILE: str = os.environ.get('SINGLE_BL_FILE',
    '/lustre/aoc/projects/hera/h6c-analysis/IDR3/lststack-outputs-131-nights/single_bl_reinpaint_1000ns/zen.LST.baseline.0_4.sum.uvh5')

# Load all knobs from the toml and inject into globals (uppercased).
toml_options = toml.load(TOML_FILE)
for toml_section in ['GLOBAL_OPTS', 'AUTO_SMOOTH_OPTS', 'SKY_MODEL_OPTS']:
    print(f'Loading config from [{toml_section}] in {TOML_FILE}:')
    for key, val in toml_options[toml_section].items():
        globals()[key.upper()] = val
        print(f'  {key.upper()} = {val!r}')

MODEL_SUFFIX: str = os.environ.get('MODEL_SUFFIX', MODEL_SUFFIX)
OUT_MODEL_FILE: str = os.environ.get('OUT_MODEL_FILE', SINGLE_BL_FILE.replace('.uvh5', MODEL_SUFFIX))

SINGLE_BL_FILE = Path(SINGLE_BL_FILE)
AUTO_FR_SPECTRUM_FILE = Path(AUTO_FR_SPECTRUM_FILE)
OUT_MODEL_FILE = Path(OUT_MODEL_FILE)
print()
for setting in ['SINGLE_BL_FILE', 'OUT_MODEL_FILE', 'MODEL_SUFFIX']:
    print(f'{setting} = "{eval(setting)}"')

## Identify Baseline and Decide Whether to Skip

In [ ]:
# Identify the baseline and locate its sibling auto file (i==j) via parent-glob.
# (For an autocorrelation input the sibling is the file itself, which is fine: we model autos too.)
ANTPAIR = tuple(int(a) for a in re.search(r'(\d+)_(\d+)', SINGLE_BL_FILE.name).group().split('_'))
glob_pattern = SINGLE_BL_FILE.parent / SINGLE_BL_FILE.name.replace(f'{ANTPAIR[0]}_{ANTPAIR[1]}', '*')
auto_candidates = sorted(SINGLE_BL_FILE.parent.glob(glob_pattern.name))
AUTO_BL_FILE = next(f for f in auto_candidates
                    if (m := re.search(r'(\d+)_(\d+)', f.name))
                    and m.group(1) == m.group(2))

# Metadata-only HERAData for the cross file: enough to decide whether to skip this baseline.
# The cross data itself is read much later (just before fitting), so we never hold the autos
# and the (4-pol) cross data in memory at the same time.
hd = io.HERAData(str(SINGLE_BL_FILE))
bl_vec = hd.antpos[ANTPAIR[0]] - hd.antpos[ANTPAIR[1]]
bl_len = np.linalg.norm(bl_vec)

# Map each cross pol to the (auto_pol_1, auto_pol_2) whose product gives the noise variance.
AUTO_POL_PAIR = {'ee': ('ee', 'ee'), 'nn': ('nn', 'nn'), 'en': ('ee', 'nn'), 'ne': ('nn', 'ee')}

print(f'Cross file: {SINGLE_BL_FILE.name}')
print(f'Auto file:  {AUTO_BL_FILE.name}')
print(f'antpair = {ANTPAIR}, baseline length = {bl_len:.2f} m, '
      f'shape = ({len(hd.times)}, {len(hd.freqs)})')

# Decide whether to process this baseline at all. Autocorrelations are modeled too (horizon = 0,
# so the delay cut is just the buffer); we only skip baselines longer than MAX_BL_LENGTH.
SKIP_THIS_FILE = False
if bl_len > MAX_BL_LENGTH:
    print(f'Baseline length {bl_len:.2f} m > MAX_BL_LENGTH {MAX_BL_LENGTH} m. Skipping.')
    SKIP_THIS_FILE = True

## Helper Functions

In [ ]:
FR_CENTER_AND_HW_CACHE = {}

def cache_fr_center_and_hw(hd, antpair, tslice, band):
    '''Compute the sky fringe-rate center and half-width (in Hz) for a given band and tslice,
    buffered by the size of the autocorrelation FR kernel, and cache it in FR_CENTER_AND_HW_CACHE.'''
    if (tslice is not None) and (band is not None) and ((antpair, tslice, band) not in FR_CENTER_AND_HW_CACHE):
        fr_buffer = get_FR_buffer_from_spectra(str(AUTO_FR_SPECTRUM_FILE), hd.times[tslice], hd.freqs[band],
                                               gauss_fit_buffer_cut=GAUSS_FIT_BUFFER_CUT)
        hd_here = hd.select(inplace=False, frequencies=hd.freqs[band])
        key = antpair + ('ee',)  # FR geometry is polarization-independent
        centers, hws = sky_frates(hd_here, keys=[key])
        fr_center = centers[key] / 1e3  # mHz -> Hz
        fr_hw = (hws[key] + np.max(fr_buffer)) / 1e3  # mHz -> Hz
        FR_CENTER_AND_HW_CACHE[(antpair, tslice, band)] = fr_center, fr_hw

## Smooth Autos via 2D DPSS

The autos have non-trivial spectral and temporal structure that, if used directly as the noise scale, leaves wiggly residuals. We 2D-DPSS-smooth each auto polarization (with a two-stage blacklist to reject outliers) and use the smooth model in the inverse-variance weights. This block is shared with `single_lststack_baseline_pI_FRF_SNR.ipynb`.

In [ ]:
# Two-stage auto-smoothing knobs
def smooth_one_auto(auto_wf, nsamp_wf, dc, tslices, bands,
                    inpaint_delay_ns=AUTO_INPAINT_DELAY,
                    blacklist_rel_err_thresh=AUTO_BLACKLIST_REL_ERR_THRESH,
                    blacklist_smoother_factor=AUTO_BLACKLIST_SMOOTHER_FACTOR,
                    blacklist_downweight=AUTO_BLACKLIST_DOWNWEIGHT,
                    eigenval_cutoff=EIGENVAL_CUTOFF,
                    cg_tol=AUTO_CG_TOL,
                    cg_iter_lim=AUTO_CG_ITER_LIM):
    '''Fit a 2D DPSS model to one auto polarization in two stages.

    Stage 1 (coarse, at delay = inpaint_delay_ns * blacklist_smoother_factor and FR halfwidth
    scaled by the same factor):
      (a) Fit with the original nsamples weights -> coarse_v0.
      (b) Build a blacklist where |raw - coarse_v0| / |coarse_v0| > blacklist_rel_err_thresh.
      (c) Re-fit at the same coarse resolution with blacklisted pixels softly downweighted -> coarse_v1.

    Stage 2 (final, at delay = inpaint_delay_ns and the full FR halfwidth):
      Replace the data at blacklisted pixels with coarse_v1, downweight those pixels by
      blacklist_downweight, then fit. Returns (smooth, blacklist).'''
    smooth = np.full_like(auto_wf, np.nan, dtype=complex)
    blacklist = np.zeros(auto_wf.shape, dtype=bool)
    times_s = (dc.times - dc.times[0]) * 86400.0  # seconds
    freqs_hz = dc.freqs

    for tslice, band in zip(tslices, bands):
        if (tslice is None) or (band is None):
            continue

        # Per-band FR halfwidth from the simulated auto FR spectrum (conservative max-over-band).
        fr_buffer = get_FR_buffer_from_spectra(
            str(AUTO_FR_SPECTRUM_FILE), dc.times[tslice], freqs_hz[band],
            gauss_fit_buffer_cut=AUTO_GAUSS_FIT_BUFFER_CUT,
        )
        fr_hw_hz = np.max(fr_buffer) / 1e3

        # Inverse-variance weights for an auto: var(V_ii) ~ V_ii^2 / nsamples, so weight ~ nsamples / V_ii^2.
        with np.errstate(invalid='ignore', divide='ignore'):
            weights = np.where(nsamp_wf[tslice, band] > 0,
                               nsamp_wf[tslice, band] / auto_wf[tslice, band]**2,
                               0).astype(float)
        if not np.any(weights > 0):
            continue

        # ---- Stage 1: coarse pre-fit at scaled delay & FR halfwidth ----
        coarse_freq_basis, _ = dpss_operator(
            freqs_hz[band], [0.0], [inpaint_delay_ns * blacklist_smoother_factor / 1e9],
            eigenval_cutoff=[eigenval_cutoff],
        )
        coarse_time_basis, _ = dpss_operator(
            times_s[tslice], [0.0], [fr_hw_hz * blacklist_smoother_factor],
            eigenval_cutoff=[eigenval_cutoff],
        )
        coarse_coeffs_v0, _ = sparse_linear_fit_2D(
            data=auto_wf[tslice, band],
            weights=weights,
            axis_1_basis=coarse_time_basis,
            axis_2_basis=coarse_freq_basis,
            precondition_solver=True,
            atol=cg_tol, btol=cg_tol, iter_lim=cg_iter_lim,
        )
        coarse_v0 = np.abs(coarse_time_basis @ coarse_coeffs_v0 @ coarse_freq_basis.T)

        # Blacklist: relative error of raw against coarse fit.
        with np.errstate(invalid='ignore', divide='ignore'):
            rel_err = np.abs(auto_wf[tslice, band] - coarse_v0) / np.abs(coarse_v0)
        bl_local = (np.isfinite(rel_err) & (rel_err > blacklist_rel_err_thresh)
                    & (nsamp_wf[tslice, band] > 0))
        blacklist[tslice, band] = bl_local

        # Coarse re-fit: same basis, but downweight the blacklisted pixels so we get a clean smoother model
        # we can use to fill them in for the final fit.
        weights_coarse_v1 = weights * np.where(bl_local, blacklist_downweight, 1.0)
        if not np.any(weights_coarse_v1 > 0):
            smooth[tslice, band] = coarse_v0
            continue
        coarse_coeffs_v1, _ = sparse_linear_fit_2D(
            data=auto_wf[tslice, band],
            weights=weights_coarse_v1,
            axis_1_basis=coarse_time_basis,
            axis_2_basis=coarse_freq_basis,
            precondition_solver=True,
            atol=cg_tol, btol=cg_tol, iter_lim=cg_iter_lim,
        )
        coarse_v1 = np.abs(coarse_time_basis @ coarse_coeffs_v1 @ coarse_freq_basis.T)

        # ---- Stage 2: final fit at full delay & FR halfwidth ----
        # Plug coarse_v1 in at blacklisted pixels and downweight them so they're fit but not trusted.
        data_final = np.where(bl_local, coarse_v1, auto_wf[tslice, band])
        weights_final = weights * np.where(bl_local, blacklist_downweight, 1.0)
        freq_basis, _ = dpss_operator(
            freqs_hz[band], [0.0], [inpaint_delay_ns / 1e9],
            eigenval_cutoff=[eigenval_cutoff],
        )
        time_basis, _ = dpss_operator(
            times_s[tslice], [0.0], [fr_hw_hz],
            eigenval_cutoff=[eigenval_cutoff],
        )
        final_coeffs, _ = sparse_linear_fit_2D(
            data=data_final,
            weights=weights_final,
            axis_1_basis=time_basis,
            axis_2_basis=freq_basis,
            precondition_solver=True,
            atol=cg_tol, btol=cg_tol, iter_lim=cg_iter_lim,
        )
        smooth[tslice, band] = np.abs(time_basis @ final_coeffs @ freq_basis.T)

    return smooth, blacklist

if not SKIP_THIS_FILE:
    # Load the sibling autos, 2D-DPSS-smooth them (or load the cache), keep only the smoothed
    # autos / blacklist / raw |auto| (for the diagnostic), then free the heavy auto objects before
    # the cross data is read.
    hd_auto = io.HERAData(str(AUTO_BL_FILE))
    autos, auto_flags, auto_nsamples = hd_auto.read(times=hd.times, polarizations=['ee', 'nn'])
    auto_antpair = sorted(set([k[:2] for k in autos.bls() if k[0] == k[1]]))[0]

    # Cache key: TOML + auto file basename. Mirrors single_lststack_baseline_pI_FRF_SNR.ipynb.
    cache_key = hashlib.md5((toml.dumps(toml_options) + AUTO_BL_FILE.name).encode()).hexdigest()
    auto_cache_file = OUT_MODEL_FILE.parent / f'smooth_autos_cache_{cache_key}.npz'

    auto_smooth, auto_blacklist = {}, {}
    if auto_cache_file.exists():
        print(f'Loading cached smoothed autos from {auto_cache_file}')
        with np.load(auto_cache_file) as cache:
            assert cache['toml_hash'].item() == cache_key, 'Cache hash mismatch'
            for pol in ('ee', 'nn'):
                auto_smooth[pol] = cache[f'auto_smooth_{pol}']
                auto_blacklist[pol] = cache[f'auto_blacklist_{pol}']
    else:
        print(f'No cache found at {auto_cache_file}, smoothing autos...')
        # Minimal (time, frequency) slices for the autos (split at FM), from the auto flags.
        for pol in ('ee', 'nn'):
            bl = auto_antpair + (pol,)
            auto_tslices, auto_bands = flag_utils.get_minimal_slices(
                auto_flags[bl], freqs=autos.freqs, freq_cuts=[(FM_LOW_FREQ + FM_HIGH_FREQ) * .5e6])
            print(f'Smoothing auto {bl}...')
            auto_smooth[pol], auto_blacklist[pol] = smooth_one_auto(
                np.abs(autos[bl]), auto_nsamples[bl], autos, auto_tslices, auto_bands)
            n_bl = int(np.sum(auto_blacklist[pol]))
            n_unflagged = int(np.sum(auto_nsamples[bl] > 0))
            if n_unflagged > 0:
                print(f'  Blacklisted {n_bl} of {n_unflagged} unflagged pixels '
                      f'({100 * n_bl / n_unflagged:.2f}%) at rel_err > {AUTO_BLACKLIST_REL_ERR_THRESH}.')

        # Atomic write to handle parallel race conditions (parallel baselines share the auto cache).
        tmp_fd, tmp_path = tempfile.mkstemp(dir=str(OUT_MODEL_FILE.parent), suffix='.npz')
        os.close(tmp_fd)
        np.savez(tmp_path, toml_hash=cache_key,
                 **{f'auto_smooth_{pol}': auto_smooth[pol] for pol in ('ee', 'nn')},
                 **{f'auto_blacklist_{pol}': auto_blacklist[pol] for pol in ('ee', 'nn')})
        try:
            os.rename(tmp_path, auto_cache_file)
            print(f'Saved smoothed autos cache to {auto_cache_file}')
        except OSError:
            os.remove(tmp_path)  # another process beat us to it
            print('Cache already written by another process.')

    # Stash raw |auto| (for the diagnostic), then free the heavy auto objects before reading crosses.
    auto_raw = {pol: np.abs(autos[auto_antpair + (pol,)]) for pol in ('ee', 'nn')}
    del hd_auto, autos, auto_flags, auto_nsamples

In [ ]:
def plot_smoothed_auto(pol):
    raw = auto_raw[pol]
    smooth = np.abs(auto_smooth[pol])
    blk = auto_blacklist[pol]
    with np.errstate(invalid='ignore', divide='ignore'):
        rel_err = np.abs(raw - smooth) / smooth
    extent = [hd.freqs[0] / 1e6, hd.freqs[-1] / 1e6,
              hd.times[-1] - int(hd.times[0]), hd.times[0] - int(hd.times[0])]
    fig, axes = plt.subplots(1, 4, figsize=(20, 5), dpi=120, sharex=True, sharey=True)
    im0 = axes[0].imshow(raw, aspect='auto', cmap='viridis', extent=extent, interpolation='none', norm=matplotlib.colors.LogNorm())
    axes[0].set_title(f'{pol} raw |auto|')
    plt.colorbar(im0, ax=axes[0])
    im1 = axes[1].imshow(smooth, aspect='auto', cmap='viridis', extent=extent, interpolation='none',
                         norm=matplotlib.colors.LogNorm(vmin=im0.get_clim()[0], vmax=im0.get_clim()[1]))
    axes[1].set_title(f'{pol} smooth |auto|')
    plt.colorbar(im1, ax=axes[1])
    im2 = axes[2].imshow(rel_err, aspect='auto', cmap='inferno', extent=extent, interpolation='none',
                         vmin=0, vmax=2 * AUTO_BLACKLIST_REL_ERR_THRESH)
    axes[2].set_title(f'{pol} |raw - smooth| / smooth')
    plt.colorbar(im2, ax=axes[2], extend='max')
    im3 = axes[3].imshow(blk.astype(int), aspect='auto', cmap='gray_r', extent=extent, interpolation='none',
                         vmin=0, vmax=1)
    axes[3].set_title(f'{pol} blacklist (rel_err > {AUTO_BLACKLIST_REL_ERR_THRESH})')
    plt.colorbar(im3, ax=axes[3])
    for ax in axes:
        ax.set_xlabel('Frequency (MHz)')
    axes[0].set_ylabel(f'JD - {int(hd.times[0])}')
    plt.tight_layout()
    plt.show()

# *Figure 1: Smoothed-Auto Diagnostic*

Per polarization: raw auto magnitude, smoothed auto magnitude, relative residual, and the blacklist mask.

In [ ]:
if not SKIP_THIS_FILE:
    for pol in ('ee', 'nn'):
        plot_smoothed_auto(pol)

## Build the Sky Model via 2D DPSS Fit

For each polarization and each (time-slice, band), we fit a 2D DPSS model whose frequency-axis (delay) half-width is the **baseline horizon delay plus a buffer**, and whose time-axis (fringe-rate) window is the sky FR center/half-width plus the autocorrelation FR buffer. We **keep the fit** `d_mdl` as the sky model. Weights are inverse-variance using the smoothed autos and nsamples; pixels with `nsamples == 0` (100% inpainted) get zero weight but are still filled in by the smooth model.

In [ ]:
if not SKIP_THIS_FILE:
    # Now read the cross-pol data (all four pols) and find each pol's minimal (time, freq) slices.
    data, flags, nsamples = hd.read()
    cross_bls = [ANTPAIR + (pol,) for pol in data.pols() if ANTPAIR + (pol,) in data]
    print(f'cross pols = {[bl[2] for bl in cross_bls]}')
    tslices, bands = {}, {}
    for bl in cross_bls:
        tslices[bl], bands[bl] = flag_utils.get_minimal_slices(
            flags[bl], freqs=data.freqs, freq_cuts=[(FM_LOW_FREQ + FM_HIGH_FREQ) * .5e6])

    # Initialize model to zeros, flagged everywhere; we'll unflag the fitted (tslice, band) boxes.
    model = {bl: np.zeros_like(data[bl]) for bl in cross_bls}
    model_flags = {bl: np.ones(data[bl].shape, dtype=bool) for bl in cross_bls}

    # Precompute FR centers/half-widths for each (tslice, band).
    for bl in cross_bls:
        for tslice, band in zip(tslices[bl], bands[bl]):
            cache_fr_center_and_hw(hd, ANTPAIR, tslice, band)

    # Delay (frequency-axis) DPSS half-width from the horizon + buffer.
    tau_horizon_ns = bl_len / constants.c * 1e9
    filter_delay_ns = tau_horizon_ns + DELAY_BUFFER
    print(f'Horizon delay = {tau_horizon_ns:.1f} ns; filter delay (horizon + {DELAY_BUFFER} ns buffer) = {filter_delay_ns:.1f} ns')

    for bl in cross_bls:
        pol = bl[2]
        # Noise scale for this cross pol: sqrt of the product of the two relevant smoothed autos.
        ap1, ap2 = AUTO_POL_PAIR[pol]
        a_smooth = np.sqrt(np.abs(auto_smooth[ap1] * auto_smooth[ap2]))
        # Inverse-variance weights; zero where nsamples == 0 (100% inpainted) or flagged.
        with np.errstate(invalid='ignore', divide='ignore'):
            wgts = np.where((nsamples[bl] > 0) & ~flags[bl], nsamples[bl] / a_smooth**2, 0).astype(float)
        if np.any(wgts > 0):
            wgts /= np.mean(wgts[wgts > 0])

        for tslice, band in zip(tslices[bl], bands[bl]):
            if (band is None) or not np.any(wgts[tslice, band] > 0):
                continue
            fr_center, fr_hw = FR_CENTER_AND_HW_CACHE[(ANTPAIR, tslice, band)]
            time_filters, _ = dpss_operator((data.times[tslice] - data.times[tslice][0]) * 3600 * 24,
                                             [fr_center], [fr_hw], eigenval_cutoff=[EIGENVAL_CUTOFF])
            freq_filters, _ = dpss_operator(data.freqs[band], [0.0], [filter_delay_ns / 1e9],
                                            eigenval_cutoff=[EIGENVAL_CUTOFF])
            fit, meta = sparse_linear_fit_2D(
                data=data[bl][tslice, band],
                weights=wgts[tslice, band],
                axis_1_basis=time_filters,
                axis_2_basis=freq_filters,
                precondition_solver=True,
                iter_lim=CG_ITER_LIM,
            )
            d_mdl = time_filters.dot(fit).dot(freq_filters.T)
            model[bl][tslice, band] = d_mdl
            model_flags[bl][tslice, band] = False
    print('Done building sky model.')

In [ ]:
def four_pol_data_vs_model_figure():
    '''One row per cross pol, four columns: phase(data), phase(model), |data|, |model|
    (|V| on a shared LogNorm). Data pixels are masked by flags; model pixels by model_flags.'''
    npols = len(cross_bls)
    fig, axes = plt.subplots(npols, 4, figsize=(16, 4 * npols), sharex=True, sharey=True, dpi=200,
                             gridspec_kw={'wspace': 0.02, 'hspace': 0.04}, squeeze=False)

    with np.errstate(invalid='ignore'):
        finite_amps = []
        for bl in cross_bls:
            finite_amps.append(np.where(~flags[bl] & (np.abs(data[bl]) > 0), np.abs(data[bl]), np.nan))
            finite_amps.append(np.where(~model_flags[bl] & (np.abs(model[bl]) > 0), np.abs(model[bl]), np.nan))
        finite_amps = [a for a in finite_amps if np.any(np.isfinite(a))]
        vmin = np.nanmin([np.nanmin(a) for a in finite_amps]) if finite_amps else np.nan
        vmax = np.nanmax([np.nanmax(a) for a in finite_amps]) if finite_amps else np.nan
    if not (np.isfinite(vmin) and np.isfinite(vmax) and vmin > 0 and vmax > vmin):
        plt.close(fig)
        print(f'No finite unflagged |V|; skipping Figure 2 (vmin={vmin}, vmax={vmax}).')
        return
    lognorm = matplotlib.colors.LogNorm(vmin=vmin, vmax=vmax)

    lst_grid = data.lsts * 12 / np.pi
    lst_grid[lst_grid > lst_grid[-1]] -= 24
    frac_jds = data.times - int(data.times[0])
    extent = [data.freqs[0] / 1e6, data.freqs[-1] / 1e6, frac_jds[-1], frac_jds[0]]

    for row, bl in zip(axes, cross_bls):
        row[0].imshow(np.where(flags[bl], np.nan, np.angle(data[bl])),
                      aspect='auto', interpolation='none', cmap='twilight', extent=extent, vmin=-np.pi, vmax=np.pi)
        row[1].imshow(np.where(model_flags[bl], np.nan, np.angle(model[bl])),
                      aspect='auto', interpolation='none', cmap='twilight', extent=extent, vmin=-np.pi, vmax=np.pi)
        row[2].imshow(np.where(flags[bl], np.nan, np.abs(data[bl])),
                      aspect='auto', interpolation='none', norm=lognorm, extent=extent)
        im = row[3].imshow(np.where(model_flags[bl], np.nan, np.abs(model[bl])),
                           aspect='auto', interpolation='none', norm=lognorm, extent=extent)
        row[0].set_ylabel(f'JD - {int(data.times[0])}')
        for ax in row:
            ax.tick_params(axis='x', direction='in')
        row[0].text(0.02, 0.99, bl, transform=row[0].transAxes, ha='left', va='top',
                    fontsize=12, color='white', bbox=dict(facecolor='black', alpha=0.5, pad=2))

    for ax, title in zip(axes[0], ['Phase (data)', 'Phase (model)', '|data|', '|model|']):
        ax.set_title(title)

    # LST right axis on the last row's rightmost panel
    ax2 = axes[-1][-1].twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')

    for ax in axes[-1]:
        ax.set_xlabel('Frequency (MHz)')

    plt.tight_layout()
    plt.colorbar(im, ax=axes.ravel().tolist(), location='right', label='|V| (Jy)', aspect=40, pad=0.04)
    plt.close(fig)
    return fig

# *Figure 2: 4-Pol Phase and Amplitude Waterfalls — Data and Sky Model*

One row per cross polarization, four columns: phase of the data, phase of the sky model, |data|, and |sky model| (on a shared LogNorm). Inspired by the 4-pol figure in `single_lststack_baseline_scaffolded_and_feathered_inpainter`. Data pixels are masked by their flags; model pixels by `model_flags`.

In [ ]:
if not SKIP_THIS_FILE and not all(np.all(model_flags[bl]) for bl in cross_bls):
    display(four_pol_data_vs_model_figure())
else:
    print('Nothing to plot (skipped or fully flagged).')

In [ ]:
def delay_fr_figure(rng, tslice):
    '''Delay-fringe-rate transform of data and sky model for all cross pols over a (time, frequency)
    slice, with white lines marking the filter boundaries: dotted vertical at the baseline horizon,
    solid vertical at the delay cut (horizon + DELAY_BUFFER), and dash-dot horizontal at the sky
    fringe-rate window. Inspired by plot_delay_fringe_rate in full_day_systematics_inspect.'''
    npols = len(cross_bls)
    freqs_rng = data.freqs[rng]
    times_rng = data.times[tslice]
    band_label = f'{freqs_rng[0] / 1e6:.1f}–{freqs_rng[-1] / 1e6:.1f} MHz'
    delays = uvtools.utils.fourier_freqs(freqs_rng) * 1e9  # ns
    frates = uvtools.utils.fourier_freqs(times_rng * 24 * 3600) * 1000  # mHz

    # Filter boundaries: horizon, delay cut (horizon + buffer), and sky FR window (center +/- hw + auto buffer).
    tau_horizon_ns = bl_len / constants.c * 1e9
    tau_cut_ns = tau_horizon_ns + DELAY_BUFFER
    key = ANTPAIR + ('ee',)  # FR geometry is polarization-independent
    centers, hws = sky_frates(hd.select(inplace=False, frequencies=freqs_rng), keys=[key])
    fr_buffer = get_FR_buffer_from_spectra(str(AUTO_FR_SPECTRUM_FILE), times_rng, freqs_rng,
                                           gauss_fit_buffer_cut=GAUSS_FIT_BUFFER_CUT)
    fr_c = centers[key]                      # mHz
    fr_hw = hws[key] + np.max(fr_buffer)     # mHz

    def dfft2(arr):
        d1 = uvtools.utils.FFT(arr[tslice, rng], axis=1, taper='bh7')
        return np.abs(uvtools.utils.FFT(d1, axis=0, taper='bh7'))

    data_ffts = {bl: dfft2(data[bl]) for bl in cross_bls}
    model_ffts = {bl: dfft2(model[bl]) for bl in cross_bls}
    allabs = np.concatenate([v.ravel() for v in data_ffts.values()])
    pos = allabs[allabs > 0]
    norm = (matplotlib.colors.LogNorm(vmin=np.median(pos), vmax=np.max(pos))
            if pos.size and np.max(pos) > np.median(pos) else None)

    fig, axes = plt.subplots(npols, 2, figsize=(9, 3.2 * npols), sharex=True, sharey=True, dpi=175, squeeze=False)
    # Show at least one buffer's width in delay beyond the horizon + buffer cut on each side.
    xlim = tau_cut_ns + DELAY_BUFFER
    ylim = max(5.0, 1.5 * (abs(fr_c) + fr_hw))
    for row, bl in zip(axes, cross_bls):
        for ax, ffts in zip(row, (data_ffts, model_ffts)):
            im = ax.imshow(ffts[bl], interpolation='none', aspect='auto', cmap='turbo', norm=norm,
                           extent=[delays[0], delays[-1], frates[-1], frates[0]])
            ax.set_xlim([-xlim, xlim])
            ax.set_ylim([-ylim, ylim])
            for sign in (-1, 1):
                ax.axvline(sign * tau_horizon_ns, color='white', ls=':', lw=1)
                ax.axvline(sign * tau_cut_ns, color='white', ls='-', lw=1)
                ax.axhline(fr_c + sign * fr_hw, color='white', ls='-.', lw=1)
        row[0].set_ylabel('Fringe Rate (mHz)')
        row[0].text(0.04, 0.96, bl, transform=row[0].transAxes, ha='left', va='top', fontsize=11,
                    color='black', bbox=dict(facecolor='w', alpha=.7, boxstyle='round'))
    axes[0][0].set_title(f'Data ({band_label})')
    axes[0][1].set_title(f'Sky Model ({band_label})')

    # Legend explaining the lines, on the top-right panel.
    legend_handles = [
        matplotlib.lines.Line2D([0], [0], color='white', ls=':', lw=1, label='Horizon'),
        matplotlib.lines.Line2D([0], [0], color='white', ls='-', lw=1, label=f'Horizon + {DELAY_BUFFER:.0f} ns buffer'),
        matplotlib.lines.Line2D([0], [0], color='white', ls='-.', lw=1, label='Sky FR window'),
    ]
    axes[0][-1].legend(handles=legend_handles, loc='upper right', fontsize=8,
                       facecolor='black', edgecolor='white', labelcolor='white', framealpha=0.6)

    for ax in axes[-1]:
        ax.set_xlabel('Delay (ns)')
    plt.tight_layout()
    if norm is not None:
        plt.colorbar(im, ax=axes.ravel().tolist(), label=r'$|\widetilde{V}_{ij}|$ (Jy)', aspect=40, pad=0.04, extend='both')
    plt.close(fig)
    return fig

# *Figure 3: Data and Sky Model in Delay–Fringe-Rate Space*

Delay–fringe-rate transform of the data and the sky model for all cross pols (one figure per clean band, below/above FM). White lines mark where the filtering was done, in the style of `full_day_systematics_inspect`: solid vertical lines at the delay cut (baseline horizon + `DELAY_BUFFER`), and dash-dot horizontal lines at the sky fringe-rate window (`sky_frates` center ± half-width + auto FR buffer). The sky model should contain power only inside this box.

In [ ]:
if not SKIP_THIS_FILE and not all(np.all(model_flags[bl]) for bl in cross_bls):
    # Reuse the (time, frequency) slices found earlier, one figure per band, taking the union
    # across cross pols so every pol's fitted region is included.
    n_bands = max(len(bands[bl]) for bl in cross_bls)
    for bi in range(n_bands):
        valid = [(tslices[bl][bi], bands[bl][bi]) for bl in cross_bls
                 if bands[bl][bi] is not None and tslices[bl][bi] is not None]
        if not valid:
            continue
        tsl = slice(min(t.start for t, _ in valid), max(t.stop for t, _ in valid))
        rng = slice(min(b.start for _, b in valid), max(b.stop for _, b in valid))
        if (rng.stop - rng.start) > 10:
            display(delay_fr_figure(rng, tsl))

## Save Sky Model

In [ ]:
if not SKIP_THIS_FILE and not all(np.all(model_flags[bl]) for bl in cross_bls):
    # hd already holds the cross data we read; overwrite it with the model and write it out.
    hd.update(data=model, flags=model_flags)
    hd.history += '\nProduced by single_baseline_sky_model_2D_filter.ipynb'
    print(f'Writing sky model to {OUT_MODEL_FILE}')
    hd.write_uvh5(str(OUT_MODEL_FILE), clobber=True)
else:
    print('Nothing to write (baseline skipped or fully flagged).')

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_qm', 'hera_filters', 'hera_notebook_templates', 'pyuvdata', 'numpy']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')